# Notebook 5b — Data Augmentation: SMOTE vs CTGAN

## What changed from R01

| Change | Reason |
|---|---|
| **CTGAN is fitted on the training fold only, refitted per fold** | R01 generated synthetic data once, outside the fold loop. A generator fitted on rows that later appear in validation is leakage, and a nastier kind than an encoder because a GAN can memorise |
| No test-set scoring | GATE-1(iii). R01 scored the seed-42 partition here too |
| Synthetic rows counted and their provenance logged | Q26 — the augmentation configurations are part of the disclosed combination count |
| Fewer seeds, stated | CTGAN is the only expensive thing in this project. Say so rather than running one seed silently |

**Warning on cost.** CTGAN refitted per fold is genuinely slow — this is the
one place where compute is a real constraint. `N_CTGAN_EPOCHS` and
`CTGAN_SEEDS` are set low deliberately. Raise them and state what you used;
do not quietly fall back to fitting once outside the loop, because that is
the leak.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance)

banner("NOTEBOOK 5b — SMOTE vs CTGAN")
OUT = run_dir("notebook05b_augmentation")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

CTGAN_SEEDS = SEEDS[:2]        # cost-limited; state this in M8
N_CTGAN_EPOCHS = 100
SYNTH_TARGETS = [250, 500]     # extra minority rows per fold
print(f"seeds {CTGAN_SEEDS}, CTGAN epochs {N_CTGAN_EPOCHS}, "
      f"synthetic targets {SYNTH_TARGETS}")

try:
    from ctgan import CTGAN
    HAVE_CTGAN = True
except ImportError:
    try:
        from sdv.single_table import CTGANSynthesizer as CTGAN
        HAVE_CTGAN = True
    except ImportError:
        HAVE_CTGAN = False
        print("CTGAN unavailable. SMOTE arms will still run; record the "
              "omission in M8 rather than leaving the comparison implied.")

try:
    from imblearn.over_sampling import SMOTE
    HAVE_SMOTE = True
except ImportError:
    HAVE_SMOTE = False

In [ ]:
# ---- fold-local augmenters ---------------------------------------------
def smote_fold(X_tr, y_tr, n_extra, seed):
    n_pos = int((y_tr == 1).sum())
    target = n_pos + n_extra
    if target > int((y_tr == 0).sum()):
        target = int((y_tr == 0).sum())
    k = max(1, min(5, n_pos - 1))
    sm = SMOTE(random_state=seed, k_neighbors=k,
               sampling_strategy={1: target, 0: int((y_tr == 0).sum())})
    Xr, yr = sm.fit_resample(X_tr, y_tr)
    return pd.DataFrame(Xr, columns=X_tr.columns), pd.Series(yr)


def ctgan_fold(X_tr, y_tr, n_extra, seed, epochs=N_CTGAN_EPOCHS):
    """Fit CTGAN on the MINORITY ROWS OF THIS TRAINING FOLD ONLY."""
    minority = X_tr[y_tr == 1].reset_index(drop=True)
    if len(minority) < 10:
        return X_tr, y_tr
    np.random.seed(seed)
    model = CTGAN(epochs=epochs, verbose=False)
    model.fit(minority)                           # training-fold rows only
    synth = model.sample(n_extra)
    synth = synth[X_tr.columns]
    Xa = pd.concat([X_tr, synth], ignore_index=True)
    ya = pd.concat([pd.Series(y_tr.to_numpy()),
                    pd.Series(np.ones(len(synth), dtype=int))], ignore_index=True)
    return Xa, ya


CONFIGS = [("original", None, 0)]
if HAVE_SMOTE:
    CONFIGS += [(f"SMOTE+{n}", "smote", n) for n in SYNTH_TARGETS]
if HAVE_CTGAN:
    CONFIGS += [(f"CTGAN+{n}", "ctgan", n) for n in SYNTH_TARGETS]
print("configurations:", [c[0] for c in CONFIGS])
print(f"total model fits: {len(CONFIGS)} x {len(CTGAN_SEEDS)} x "
      f"{N_SPLITS*N_REPEATS} = {len(CONFIGS)*len(CTGAN_SEEDS)*N_SPLITS*N_REPEATS}")
print("Add this to the disclosed combination count in M13 (Q26).")

In [ ]:
# ---- run ----------------------------------------------------------------
from lightgbm import LGBMClassifier
import time

rows, prov = [], []
t0 = time.perf_counter()
for seed in CTGAN_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        for label, kind, n_extra in CONFIGS:
            if kind is None:
                Xa, ya = X_tr, y_tr
            elif kind == "smote":
                Xa, ya = smote_fold(X_tr, y_tr, n_extra, seed)
            else:
                Xa, ya = ctgan_fold(X_tr, y_tr, n_extra, seed)

            model = LGBMClassifier(objective="binary", random_state=seed,
                                   **SHARED_PARAMS)
            model.fit(Xa, ya)
            p = model.predict_proba(X_vl)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": label,
                         "augmentation": kind or "none",
                         "n_synthetic": len(ya) - len(y_tr),
                         "n_train_rows": len(ya),
                         **score_binary(y_vl, p)})
            prov.append({"seed": seed, "fold": fi, "config": label,
                         "generator_fitted_on": "training fold minority rows only",
                         "n_synthetic_rows": len(ya) - len(y_tr),
                         "validation_rows_synthetic": 0,
                         "validation_rows": len(y_vl)})
        print(f"    seed {seed} fold {fi}/{N_SPLITS*N_REPEATS} "
              f"[{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "augmentation_fold_scores.csv", index=False)
pd.DataFrame(prov).to_csv(OUT / "synthetic_data_provenance.csv", index=False)
print(f"\n{len(fold_df)} fold-level rows")
print("PROVENANCE: every generator was fitted on training-fold rows only, and "
      "zero synthetic rows entered any validation fold.")

In [ ]:
# ---- results ------------------------------------------------------------
var = two_level_variance(fold_df, PRIMARY_METRIC)
summary = (fold_df.groupby(["arm", "augmentation"])
           .agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"),
                recall=("recall", "mean"), precision=("precision", "mean"),
                n_synthetic=("n_synthetic", "mean"))
           .reset_index()
           .merge(var[["arm", "between_seed_sd"]], on="arm")
           .sort_values("auc_pr", ascending=False))
summary.to_csv(OUT / "augmentation_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

base = summary[summary["arm"] == "original"]["auc_pr"].iloc[0]
print(f"\nvs the unaugmented training distribution ({base:.4f}):")
for _, r in summary.iterrows():
    if r["arm"] != "original":
        d = r["auc_pr"] - base
        flag = "" if abs(d) > r["between_seed_sd"] else "  (smaller than its own SD)"
        print(f"  {r['arm']:14s} {d:+.4f}{flag}")

plt.figure(figsize=(8, 4))
o = summary.sort_values("auc_pr")
plt.barh(o["arm"], o["auc_pr"], xerr=o["between_seed_sd"], color="steelblue")
plt.axvline(base, ls="--", c="k", lw=1, label="unaugmented")
plt.xlabel("AUC-PR"); plt.legend(); plt.tight_layout()
plt.savefig(OUT / "figures/augmentation_comparison.png", dpi=200); plt.close()

print("\nR10 reported that neither SMOTE nor CTGAN improved on the raw "
      "distribution. If that survives fold-local generator fitting, it is a "
      "stronger result than before, and it is the correct place to say so.")

write_manifest(OUT, {"notebook": "05b_augmentation", "test_set_scored": False,
                     "seeds": CTGAN_SEEDS, "ctgan_epochs": N_CTGAN_EPOCHS,
                     "configurations": [c[0] for c in CONFIGS],
                     "generator_scope": "training fold only, refitted per fold",
                     "total_fits": int(len(fold_df))})